# Notebook 2: From Embeddings to Topics

This notebook takes the document embeddings generated in Notebook 1 and runs the remaining BERTopic pipeline steps manually:

1. **Dimensionality Reduction** (UMAP) — compress 384/768/1536-D embeddings to 5-D
2. **Clustering** (HDBSCAN) — discover dense groups of semantically similar documents
3. **Topic Representation** (c-TF-IDF) — extract interpretable keywords per cluster

We then compare discovered topics against the curated human labels from earlier assignments.

**Prerequisites:** Run Notebook 1 first to generate `.npy` embedding files.

---

## 1. Setup & Load Saved Embeddings

In [ ]:
# Install required packages (uncomment as needed)
# !pip install umap-learn hdbscan pandas numpy scikit-learn matplotlib seaborn plotly

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
print("Libraries loaded.")

In [ ]:
# Load the original dataset with metadata and labels
DATA_PATH = "english_pages_metadata_clean.csv"
LABELS_PATH = "english_pages_metadata_clean_with_labels.csv"

df = pd.read_csv(DATA_PATH, encoding='utf-8-sig')
df['full_text'] = df['full_text'].fillna('').astype(str)

# Merge labels
if os.path.exists(LABELS_PATH):
    labels_df = pd.read_csv(LABELS_PATH)
    labels_subset = labels_df[['page_id', 'manual_label_final']].copy()
    df = pd.merge(df, labels_subset, on='page_id', how='left')
    print(f"Labels available: {df['manual_label_final'].notna().sum()} labeled documents")
    print(f"Categories: {sorted(df['manual_label_final'].dropna().unique())}")

print(f"\nLoaded {len(df)} documents.")

In [ ]:
# Load saved embeddings from Notebook 1
# Choose which embedding model to use for the pipeline
# You can re-run this notebook with different embeddings to compare results

available_embeddings = {}

for name, path in [('MiniLM (384-D)', 'minilm_embeddings.npy'),
                    ('mpnet (768-D)', 'mpnet_embeddings.npy'),
                    ('OpenAI (1536-D)', 'openai_embeddings.npy'),
                    ('Google (768-D)', 'google_embeddings.npy')]:
    if os.path.exists(path):
        emb = np.load(path)
        available_embeddings[name] = emb
        print(f"  ✓ Loaded {name}: {emb.shape}")
    else:
        print(f"  ✗ Not found: {path}")

print(f"\n{len(available_embeddings)} embedding model(s) available.")

In [ ]:
# ============================================================
# SELECT YOUR EMBEDDING MODEL HERE
# Change this to try different models through the pipeline
# ============================================================
SELECTED_MODEL = 'MiniLM (384-D)'  # BERTopic's default

embeddings = available_embeddings[SELECTED_MODEL]
print(f"Selected: {SELECTED_MODEL}")
print(f"Embedding matrix: {embeddings.shape}")
print(f"Documents: {embeddings.shape[0]}, Dimensions: {embeddings.shape[1]}")

---
# Step 1: Dimensionality Reduction with UMAP
---

**Why reduce dimensions?** Clustering algorithms like HDBSCAN struggle in high-dimensional spaces due to the curse of dimensionality — distances between all points converge, making it impossible to distinguish dense regions from sparse ones.

**UMAP** (Uniform Manifold Approximation and Projection) compresses high-dimensional data while preserving local neighborhood structure. Points that are close in the original space stay close in the reduced space.

We reduce to:
- **2 dimensions** for visualization
- **5 dimensions** for clustering (retains more information)

In [ ]:
from umap import UMAP

print("Reducing to 2-D for visualization...")
umap_2d = UMAP(
    n_components=2,
    n_neighbors=15,     # size of local neighborhood
    min_dist=0.0,       # how tightly points can pack (0 = very tight clusters)
    metric='cosine',    # cosine works best for normalized embeddings
    random_state=42
)
reduced_2d = umap_2d.fit_transform(embeddings)
print(f"  2-D shape: {reduced_2d.shape}")

print("\nReducing to 5-D for clustering...")
umap_5d = UMAP(
    n_components=5,
    n_neighbors=15,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)
reduced_5d = umap_5d.fit_transform(embeddings)
print(f"  5-D shape: {reduced_5d.shape}")

print(f"\nDimensionality: {embeddings.shape[1]} → 5 (for clustering) and → 2 (for plotting)")

### Visualize the 2-D Projection

If we have curated labels, color the points by human-assigned category. Do the categories visually separate?

In [ ]:
# 2-D scatter plot colored by curated labels
vis_df = pd.DataFrame(reduced_2d, columns=['x', 'y'])

label_col = 'manual_label_final'
if label_col in df.columns:
    vis_df['label'] = df[label_col].fillna('Unlabeled')
    
    # Plot labeled documents only (for clarity)
    labeled = vis_df[vis_df['label'] != 'Unlabeled']
    unlabeled = vis_df[vis_df['label'] == 'Unlabeled']
    
    fig, ax = plt.subplots(figsize=(14, 10))
    
    # Plot unlabeled as grey background
    if len(unlabeled) > 0:
        ax.scatter(unlabeled['x'], unlabeled['y'], c='lightgrey', s=5, alpha=0.3, label='Unlabeled')
    
    # Plot labeled with colors
    categories = sorted(labeled['label'].unique())
    colors = plt.cm.Set2(np.linspace(0, 1, len(categories)))
    for cat, color in zip(categories, colors):
        mask = labeled['label'] == cat
        ax.scatter(labeled.loc[mask, 'x'], labeled.loc[mask, 'y'], 
                   c=[color], s=20, alpha=0.7, label=cat)
    
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
    ax.set_title(f'UMAP 2-D Projection — {SELECTED_MODEL}\nColored by Curated Human Labels', fontsize=14)
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    plt.tight_layout()
    plt.show()
    
    print("\nQuestion: Do the curated categories form distinct visual clusters?")
    print("Where do categories overlap? What might that tell us?")
else:
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.scatter(vis_df['x'], vis_df['y'], c='steelblue', s=5, alpha=0.3)
    ax.set_title(f'UMAP 2-D Projection — {SELECTED_MODEL}', fontsize=14)
    plt.tight_layout()
    plt.show()

### Experiment: UMAP Parameters

How do different UMAP settings affect the projection?

In [ ]:
# Compare different n_neighbors values
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for idx, n_neigh in enumerate([5, 15, 50]):
    umap_temp = UMAP(n_components=2, n_neighbors=n_neigh, min_dist=0.0, 
                     metric='cosine', random_state=42)
    reduced_temp = umap_temp.fit_transform(embeddings)
    
    axes[idx].scatter(reduced_temp[:, 0], reduced_temp[:, 1], 
                      c='steelblue', s=3, alpha=0.3)
    axes[idx].set_title(f'n_neighbors={n_neigh}', fontsize=13)
    axes[idx].set_xlabel('UMAP 1')
    axes[idx].set_ylabel('UMAP 2')

plt.suptitle(f'Effect of n_neighbors on UMAP — {SELECTED_MODEL}', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Low n_neighbors → more local structure, tighter micro-clusters")
print("High n_neighbors → more global structure, broader groupings")

---
# Step 2: Clustering with HDBSCAN
---

**HDBSCAN** (Hierarchical Density-Based Spatial Clustering of Applications with Noise) finds dense regions of points and groups them into clusters.

**Key advantages over K-Means:**
- No need to specify the number of clusters
- Finds clusters of varying shapes and sizes
- Labels ambiguous/isolated points as outliers (topic -1)

**Key parameters:**
- `min_cluster_size` — minimum number of documents to form a topic
- `min_samples` — how conservative the density estimation is

In [ ]:
from hdbscan import HDBSCAN

# Cluster the 5-D reduced embeddings
hdbscan_model = HDBSCAN(
    min_cluster_size=15,    # minimum docs to form a topic
    min_samples=5,          # density sensitivity
    metric='euclidean',
    cluster_selection_method='eom'  # Excess of Mass (default)
)

clusters = hdbscan_model.fit_predict(reduced_5d)

n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
n_outliers = (clusters == -1).sum()
pct_outliers = n_outliers / len(clusters) * 100

print(f"Clustering Results:")
print(f"  Topics discovered: {n_clusters}")
print(f"  Outliers (topic -1): {n_outliers} ({pct_outliers:.1f}%)")
print(f"  Documents clustered: {len(clusters) - n_outliers}")

In [ ]:
# Cluster size distribution
cluster_counts = pd.Series(clusters).value_counts().sort_index()

print(f"\n{'Cluster':<10} {'Size':>8} {'Percentage':>12}")
print("-" * 32)
for cluster_id, count in cluster_counts.items():
    label = 'Outliers' if cluster_id == -1 else f'Topic {cluster_id}'
    print(f"{label:<10} {count:>8} {count/len(clusters)*100:>10.1f}%")

In [ ]:
# Visualize clusters on 2-D UMAP projection
vis_df['cluster'] = clusters

fig, ax = plt.subplots(figsize=(14, 10))

# Plot outliers as grey background
outliers = vis_df[vis_df['cluster'] == -1]
clustered = vis_df[vis_df['cluster'] != -1]

ax.scatter(outliers['x'], outliers['y'], c='lightgrey', s=5, alpha=0.2, label='Outliers')

# Plot each cluster with a distinct color
unique_clusters = sorted(clustered['cluster'].unique())
colors = plt.cm.tab20(np.linspace(0, 1, max(len(unique_clusters), 1)))

for cl, color in zip(unique_clusters, colors):
    mask = clustered['cluster'] == cl
    ax.scatter(clustered.loc[mask, 'x'], clustered.loc[mask, 'y'],
               c=[color], s=15, alpha=0.6, label=f'Topic {cl}')

ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8, ncol=2)
ax.set_title(f'HDBSCAN Clusters on UMAP 2-D — {SELECTED_MODEL}\n{n_clusters} topics discovered, {pct_outliers:.1f}% outliers', fontsize=14)
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
plt.tight_layout()
plt.show()

### Experiment: HDBSCAN Parameters

How does `min_cluster_size` affect the number and quality of topics?

In [ ]:
# Sweep min_cluster_size
print(f"{'min_cluster_size':>18} {'Topics':>8} {'Outliers':>10} {'Outlier %':>10}")
print("-" * 50)

for mcs in [5, 10, 15, 25, 50, 100]:
    hdb_temp = HDBSCAN(min_cluster_size=mcs, min_samples=5, metric='euclidean')
    temp_clusters = hdb_temp.fit_predict(reduced_5d)
    n_topics = len(set(temp_clusters)) - (1 if -1 in temp_clusters else 0)
    n_out = (temp_clusters == -1).sum()
    pct = n_out / len(temp_clusters) * 100
    print(f"{mcs:>18} {n_topics:>8} {n_out:>10} {pct:>9.1f}%")

print("\nSmaller min_cluster_size → more granular topics (but possibly noisy)")
print("Larger min_cluster_size → fewer, broader topics (but more outliers)")

---
# Step 3: Topic Representation with c-TF-IDF
---

Now we know *which* documents belong to each cluster, but we don't know *what* each cluster is about. c-TF-IDF (class-based TF-IDF) solves this by:

1. Concatenating all documents in a cluster into one "class document"
2. Computing TF-IDF where each cluster is a single document
3. Ranking words by their importance within each cluster relative to all others

This gives us the top keywords that distinguish each topic.

In [ ]:
def compute_ctfidf(documents, clusters, top_n=10):
    """
    Compute c-TF-IDF: class-based TF-IDF for topic representation.
    
    Parameters:
    -----------
    documents : list of str — all document texts
    clusters : array-like — cluster assignment per document
    top_n : int — number of top keywords per topic
    
    Returns:
    --------
    dict mapping cluster_id → list of (word, score) tuples
    """
    # Group documents by cluster
    cluster_docs = {}
    for doc, cluster_id in zip(documents, clusters):
        if cluster_id == -1:
            continue  # skip outliers
        if cluster_id not in cluster_docs:
            cluster_docs[cluster_id] = []
        cluster_docs[cluster_id].append(doc)
    
    # Concatenate documents per cluster into single "class documents"
    cluster_ids = sorted(cluster_docs.keys())
    class_documents = [' '.join(cluster_docs[cid]) for cid in cluster_ids]
    
    # Fit CountVectorizer on the class documents
    vectorizer = CountVectorizer(
        max_features=10000,
        stop_words='english',
        min_df=2,
        ngram_range=(1, 2)
    )
    tf = vectorizer.fit_transform(class_documents)
    
    # c-TF-IDF computation
    # TF: word count per class / total words in class
    tf_dense = tf.toarray().astype(float)
    tf_normalized = tf_dense / tf_dense.sum(axis=1, keepdims=True)
    
    # IDF: log(1 + total docs / docs containing word)
    n_classes = len(class_documents)
    df_count = (tf_dense > 0).sum(axis=0)
    idf = np.log(1 + n_classes / (df_count + 1))
    
    # c-TF-IDF = TF * IDF
    ctfidf = tf_normalized * idf
    
    # Extract top keywords per cluster
    feature_names = vectorizer.get_feature_names_out()
    topic_keywords = {}
    
    for idx, cid in enumerate(cluster_ids):
        scores = ctfidf[idx]
        top_indices = scores.argsort()[-top_n:][::-1]
        keywords = [(feature_names[i], scores[i]) for i in top_indices]
        topic_keywords[cid] = keywords
    
    return topic_keywords

In [ ]:
# Compute c-TF-IDF for our clusters
documents = df['full_text'].tolist()
topic_keywords = compute_ctfidf(documents, clusters, top_n=10)

print(f"Topic representations generated for {len(topic_keywords)} topics.\n")
print("=" * 80)

for topic_id in sorted(topic_keywords.keys()):
    keywords = topic_keywords[topic_id]
    size = (clusters == topic_id).sum()
    keyword_str = ' | '.join([f"{word}" for word, score in keywords[:7]])
    print(f"\nTopic {topic_id} ({size} docs):")
    print(f"  Keywords: {keyword_str}")

In [ ]:
# Visualize top keywords per topic as bar charts
n_topics_to_show = min(len(topic_keywords), 12)
topic_ids = sorted(topic_keywords.keys())[:n_topics_to_show]

cols = 3
rows = (n_topics_to_show + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(18, 4 * rows))
axes = axes.flatten() if n_topics_to_show > 1 else [axes]

for idx, topic_id in enumerate(topic_ids):
    keywords = topic_keywords[topic_id][:8]
    words = [w for w, s in keywords][::-1]
    scores = [s for w, s in keywords][::-1]
    size = (clusters == topic_id).sum()
    
    axes[idx].barh(words, scores, color=plt.cm.tab20(idx / 20), edgecolor='black', linewidth=0.3)
    axes[idx].set_title(f'Topic {topic_id} ({size} docs)', fontsize=11)
    axes[idx].set_xlabel('c-TF-IDF Score')

# Hide empty subplots
for idx in range(n_topics_to_show, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle(f'Top Keywords per Topic — {SELECTED_MODEL}', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
# Comparing Discovered Topics to Curated Labels
---

The key question: **do the topics HDBSCAN discovered align with the 8 categories humans defined?**

We build a cross-tabulation (confusion-style matrix) mapping each cluster to the curated labels, and compute alignment metrics.

In [ ]:
label_col = 'manual_label_final'

if label_col in df.columns:
    # Create comparison dataframe for labeled documents only
    comp_df = pd.DataFrame({
        'cluster': clusters,
        'label': df[label_col]
    })
    comp_df = comp_df[comp_df['label'].notna()].copy()
    
    # Cross-tabulation
    cross_tab = pd.crosstab(comp_df['cluster'], comp_df['label'])
    
    print(f"Cross-tabulation: {cross_tab.shape[0]} clusters × {cross_tab.shape[1]} curated categories\n")
    print(cross_tab.to_string())
else:
    print("No labels available for comparison.")

In [ ]:
# Heatmap of cluster vs. curated label alignment
if label_col in df.columns:
    # Normalize by row (cluster) to show label distribution within each cluster
    cross_tab_norm = cross_tab.div(cross_tab.sum(axis=1), axis=0)
    
    fig, axes = plt.subplots(1, 2, figsize=(20, max(8, len(cross_tab) * 0.5)))
    
    # Raw counts
    sns.heatmap(cross_tab, annot=True, fmt='d', cmap='Blues', ax=axes[0])
    axes[0].set_title('Cluster × Label — Raw Counts', fontsize=13)
    axes[0].set_xlabel('Curated Label')
    axes[0].set_ylabel('Discovered Cluster')
    
    # Normalized (proportion within each cluster)
    sns.heatmap(cross_tab_norm, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[1])
    axes[1].set_title('Cluster × Label — Proportion Within Cluster', fontsize=13)
    axes[1].set_xlabel('Curated Label')
    axes[1].set_ylabel('Discovered Cluster')
    
    plt.suptitle(f'Discovered Topics vs. Curated Categories — {SELECTED_MODEL}', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
    
    print("\nLeft: How many labeled docs from each category landed in each cluster.")
    print("Right: Within each cluster, what fraction belongs to each category.")
    print("\nA 'pure' cluster has one dominant category (high value in one column, low elsewhere).")
    print("A 'mixed' cluster spans multiple categories — the human labels may not reflect natural structure.")

In [ ]:
# Quantitative alignment metrics
if label_col in df.columns:
    # Filter to labeled, non-outlier documents
    eval_mask = (comp_df['cluster'] != -1)
    eval_clusters = comp_df.loc[eval_mask, 'cluster'].values
    eval_labels = comp_df.loc[eval_mask, 'label'].values
    
    ari = adjusted_rand_score(eval_labels, eval_clusters)
    nmi = normalized_mutual_info_score(eval_labels, eval_clusters)
    
    print(f"Alignment Metrics (labeled, non-outlier documents):")
    print(f"  Adjusted Rand Index (ARI): {ari:.4f}")
    print(f"  Normalized Mutual Information (NMI): {nmi:.4f}")
    print(f"\n  ARI: 0 = random, 1 = perfect match")
    print(f"  NMI: 0 = no shared information, 1 = perfect correspondence")
    print(f"\n  Note: We don't expect 1.0 — the question is whether natural")
    print(f"  structure aligns with human intuitions about categories.")

### Per-Cluster Analysis: What Does Each Topic Look Like?

In [ ]:
# For each cluster, show: top keywords, dominant curated label, and sample documents
for topic_id in sorted(topic_keywords.keys())[:10]:  # show first 10
    keywords = topic_keywords[topic_id]
    size = (clusters == topic_id).sum()
    keyword_str = ', '.join([w for w, s in keywords[:6]])
    
    print(f"\n{'='*80}")
    print(f"TOPIC {topic_id} — {size} documents")
    print(f"Keywords: {keyword_str}")
    
    # Show label distribution if available
    if label_col in df.columns:
        topic_mask = clusters == topic_id
        topic_labels = df.loc[topic_mask, label_col].dropna()
        if len(topic_labels) > 0:
            print(f"Label distribution:")
            for lbl, cnt in topic_labels.value_counts().head(5).items():
                print(f"  {lbl}: {cnt} ({cnt/len(topic_labels)*100:.1f}%)")
    
    # Show 3 sample documents
    topic_docs = df.loc[clusters == topic_id]
    print(f"\nSample documents:")
    for _, row in topic_docs.head(3).iterrows():
        preview = str(row['full_text'])[:150].replace('\n', ' ')
        print(f"  [{row['page_id']}] {preview}...")

---
# Bonus: Compare Across Embedding Models
---

Run this section after you've tried the full pipeline with multiple embedding models from Notebook 1. Change `SELECTED_MODEL` at the top and re-run, or compare saved results.

In [ ]:
# Run the full pipeline for each available embedding and collect results
results_summary = []

for model_name, embs in available_embeddings.items():
    print(f"\nProcessing {model_name}...")
    
    # UMAP
    umap_temp = UMAP(n_components=5, n_neighbors=15, min_dist=0.0, 
                     metric='cosine', random_state=42)
    reduced_temp = umap_temp.fit_transform(embs)
    
    # HDBSCAN
    hdb_temp = HDBSCAN(min_cluster_size=15, min_samples=5, metric='euclidean')
    clusters_temp = hdb_temp.fit_predict(reduced_temp)
    
    n_topics = len(set(clusters_temp)) - (1 if -1 in clusters_temp else 0)
    n_outliers = (clusters_temp == -1).sum()
    pct_out = n_outliers / len(clusters_temp) * 100
    
    # Alignment with labels
    ari_val, nmi_val = 0.0, 0.0
    if label_col in df.columns:
        labeled_mask = df[label_col].notna() & (clusters_temp != -1)
        if labeled_mask.sum() > 0:
            ari_val = adjusted_rand_score(
                df.loc[labeled_mask, label_col].values,
                clusters_temp[labeled_mask]
            )
            nmi_val = normalized_mutual_info_score(
                df.loc[labeled_mask, label_col].values,
                clusters_temp[labeled_mask]
            )
    
    results_summary.append({
        'Model': model_name,
        'Dimensions': embs.shape[1],
        'Topics': n_topics,
        'Outliers': f"{n_outliers} ({pct_out:.1f}%)",
        'ARI': f"{ari_val:.4f}",
        'NMI': f"{nmi_val:.4f}"
    })

results_df = pd.DataFrame(results_summary)
print("\n" + "=" * 80)
print("EMBEDDING MODEL COMPARISON")
print("=" * 80)
print(results_df.to_string(index=False))
print("\nHigher ARI/NMI = discovered topics align more closely with curated labels.")
print("More topics = more granular (could be better or noisier).")
print("Fewer outliers = more documents assigned to a topic.")

---
## Summary

In this notebook you:

1. **Reduced dimensions** with UMAP — compressed 384/768/1536-D embeddings to 5-D for clustering and 2-D for visualization
2. **Clustered** with HDBSCAN — discovered dense groups of semantically similar documents without specifying K
3. **Labeled topics** with c-TF-IDF — extracted the most distinctive keywords per cluster to make topics interpretable
4. **Compared** discovered topics against curated human labels using cross-tabulation, ARI, and NMI

**Key takeaways:**
- You built every component of the BERTopic pipeline by hand
- Different embedding models produce different topic structures — the first step is the most impactful
- Discovered topics may not match human labels perfectly — that's the point. Clustering reveals natural structure that human intuitions may miss or oversimplify

**What's next:** In the optional Notebook 3, you can run BERTopic as a unified framework and explore advanced features like `approximate_distribution` for multi-topic documents and dynamic topic modeling over time.